# 08 — Build Your Own Metric

`judge_faithfulness` and `judge_correctness` from notebook 07 already *are* the core idea behind every metric a framework like RAGAs or DeepEval ships. The only thing left is packaging them for reuse -- a class with a `.score()` method, instead of a standalone function you have to remember to call correctly every time.

This is the notebook where "Framework != Evaluation, Framework = Automation" stops being a slogan someone else wrote and becomes something you built yourself, in about thirty lines.


In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"

DATA_DIR = os.path.join("..", "data")
with open(os.path.join(DATA_DIR, "eval_examples.json")) as f:
    examples = json.load(f)


## Packaging faithfulness

Same prompt as notebook 07, wrapped so it returns one number instead of a list of claims you have to summarize yourself every time.


In [ ]:
FAITHFULNESS_PROMPT = """You are a faithfulness judge. Extract every factual claim in the \
answer below, then mark each as "supported" or "unsupported" based only on the context. \
Return strict JSON only, no other text, in this shape:
{{"claims": [{{"text": "...", "supported": true, "reason": "..."}}]}}

Context:
{context}

Answer:
{answer}"""


class FaithfulnessMetric:
    def __init__(self, client, model: str = MODEL):
        self.client = client
        self.model = model

    def score(self, context: str, answer: str) -> tuple[float, list[dict]]:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": FAITHFULNESS_PROMPT.format(context=context, answer=answer)}],
            temperature=0.0,
            reasoning_effort="low",
        )
        result = json.loads(response.choices[0].message.content)
        claims = result["claims"]
        if not claims:
            return 1.0, claims  # nothing claimed, nothing unsupported
        supported = sum(1 for c in claims if c["supported"])
        return supported / len(claims), claims


## Packaging correctness

Same idea, mapping the three-way verdict onto a number so it can be averaged and tracked over time, the same way `faithfulness` now can be.


In [ ]:
CORRECTNESS_PROMPT = """Expected answer: {gold}
Actual answer: {answer}

Is the actual answer correct and equivalent to the expected answer? Consider partial matches -- \
an answer that supports a real but different clause than the expected one should be scored \
PARTIALLY_CORRECT, not INCORRECT. If the expected answer is "Not stated in the documents" and \
the actual answer honestly declines to answer, that counts as CORRECT.

Return strict JSON only: {{"verdict": "CORRECT|PARTIALLY_CORRECT|INCORRECT", "reason": "..."}}"""

VERDICT_SCORES = {"CORRECT": 1.0, "PARTIALLY_CORRECT": 0.5, "INCORRECT": 0.0}


class CorrectnessMetric:
    def __init__(self, client, model: str = MODEL):
        self.client = client
        self.model = model

    def score(self, gold: str, answer: str) -> tuple[float, dict]:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": CORRECTNESS_PROMPT.format(gold=gold, answer=answer)}],
            temperature=0.0,
            reasoning_effort="low",
        )
        result = json.loads(response.choices[0].message.content)
        return VERDICT_SCORES[result["verdict"]], result


## Running both metrics across all 20 examples

This is the payoff: one line per metric, per example, instead of hand-writing a prompt every time.


In [ ]:
faithfulness_metric = FaithfulnessMetric(client)
correctness_metric = CorrectnessMetric(client)

scored = []
for i, ex in enumerate(examples):
    f_score, f_claims = faithfulness_metric.score(ex["context_text"], ex["system_answer"])
    c_score, c_result = correctness_metric.score(ex["gold_answer"], ex["system_answer"])
    scored.append({"index": i, "source": ex["source"], "faithfulness": f_score, "correctness": c_score})

print(f"{'#':<4}{'Source':<10}{'Faithfulness':<14}{'Correctness'}")
for s in scored:
    print(f"{s['index']:<4}{s['source']:<10}{s['faithfulness']:<14.2f}{s['correctness']:.2f}")


In [ ]:
avg_faithfulness = sum(s["faithfulness"] for s in scored) / len(scored)
avg_correctness = sum(s["correctness"] for s in scored) / len(scored)

print(f"\nAverage faithfulness: {avg_faithfulness:.2f}")
print(f"Average correctness:  {avg_correctness:.2f}")


## Two averages are not the whole story

Notice these two numbers can move independently. A system can average high faithfulness (it rarely invents anything) while averaging lower correctness (it often declines to answer, or matches the wrong specific clause, the way example 7 did back in notebooks 06 and 07). If you only tracked one combined "quality" score, that distinction -- *conservative but accurate when it does answer* versus *confidently wrong* -- would disappear into a single number. This is exactly why production systems (notebook `11`) track metrics separately instead of collapsing everything into one score.


## Two scores that should worry you

Look back at the full table. Example `10` scored `correctness: 0.00` -- but you judged this one "good" yourself in notebook 01: the system correctly reasoned that Scott Derrickson and Ed Wood are both American. Example `17`'s *honest* answer -- the one that correctly declines to guess -- scored `faithfulness: 0.00`. Both are worth chasing down, because packaging a judge into a class didn't just make it convenient. It also made it easy to run across all 20 examples without reading a single one of the individual verdicts, which is exactly how a real bug in the metric would slip past you.


In [ ]:
result, raw = correctness_metric.score(examples[10]["gold_answer"], examples[10]["system_answer"])
print("Score:", result)
print("Reason:", raw["reason"])


There it is: the judge marked it incorrect because the gold answer is a bare `"yes"` and the system's answer never says the word "yes" -- even though "Both Scott Derrickson and Ed Wood are American" straightforwardly *means* yes. This is a real brittleness in this judge prompt, not a one-off fluke: terse yes/no gold answers get compared too literally against fuller natural-language answers that never restate the yes/no explicitly.


In [ ]:
_, raw_claims = faithfulness_metric.score(examples[17]["context_text"], examples[17]["system_answer"])
for c in raw_claims:
    print(c["supported"], "-", c["text"])
    print("  ->", c["reason"])


Even more interesting: the judge extracted the refusal itself -- *"the context does not contain information about whom King Charles III swore fealty to"* -- as a claim, then checked *that statement* against the context. And technically, it's right to flag it: the context does discuss a fealty relationship between Charles III and Rollo, just in the reverse direction from what was asked. The refusal is being held to the same "does this claim match the context" test as a positive assertion, when a refusal isn't really a fact to check -- it's a statement about the *absence* of a specific fact, which is a different thing to verify than "is this claim true."

Neither of these is a reason to throw the metric out. They're exactly the kind of thing calibration in notebook 07 exists to surface -- except this time it took running the packaged version across all 20 examples, not just the 4 you spot-checked, to actually find them. That's worth remembering the moment a metric starts feeling convenient enough to stop reading its output.


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Metric class | A packaged, reusable version of a judge prompt, with a `.score()` method |
| Framework vs. automation | A framework doesn't invent new judgment -- it automates a specific human judgment you could write yourself |

`FaithfulnessMetric` and `CorrectnessMetric` are, functionally, exactly what a framework's built-in faithfulness and correctness metrics do. Different prompt wording, maybe a different score formula, but the same underlying idea: send a structured question to a judge model, parse the structured answer, turn it into a number. Once you've built one yourself, a framework's metric class stops being a mystery and starts being a design choice you can evaluate on its own merits.

**Next up:** notebook `09` installs two real frameworks and runs their faithfulness metric on this exact same set of 20 examples -- so you're comparing them against your own homemade version, not just taking their number on faith.

**Exercises:**

- Rewrite `FAITHFULNESS_PROMPT` to explicitly tell the judge that a refusal ("the context does not address X") should be checked differently from a positive assertion -- does that fix example 17's score without breaking anything else?
- Rewrite `CORRECTNESS_PROMPT` to explicitly allow a fuller natural-language answer to count as matching a bare "yes"/"no" gold answer when it clearly implies the same thing -- re-run it on example 10 and see if that alone fixes it.
- Add a `.score_batch()` method to either class that takes a list of examples and returns all scores in one call, instead of looping externally.
